In [1]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=12"

import sys
sys.path.append('../../src/')

import numpy as onp
import jax
import jax.numpy as jnp

from myutils_2233 import Ntime, ACFs, set_detectors
from myutils_2233 import ln_likelihood_full_jit

from scipy.linalg import toeplitz
import scipy.signal as sig

import lal
from gwpy.timeseries import TimeSeries
from other_utils import bandpass_ds, analysis_data, interp1d_jax, load_tables

jax.config.update("jax_enable_x64", True) 

/Users/kallol/Work/Misc/ringdown__2/examples/fixed-sky/../../src/myutils_2233.py:3: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from corner import corner
from chainconsumer import Chain, ChainConsumer, Truth, ChainConfig, PlotConfig

import numpyro
from numpyro.contrib.nested_sampling import NestedSampler
import numpyro.distributions as dist

import h5py

# Make sure numpyro uses 64-bit too (since you enabled x64 in JAX)
numpyro.enable_x64()

/Users/kallol/miniconda3/envs/skyloc_ringdown/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tgps = 1242442967.445 + 0.006 
seglen = 8 
fs = 16384 
fmin = 8 
event_id = "GW190521" 
plot_checks = 0 
T = 0.2 
srate = 4096 

ra = 3.5 
dec = 0.73 


factor = 10
seed       = 31567
t0         = 0.0
fmax       = 1500

qnm1_path  = "../../data/l2/n1l2m2.dat"
qnm2_path  = "../../data/l3/n1l3m3.dat"

tM_shifted_samples_save_dir = "./GW190521-t-shift-posteriors/"

In [4]:
dH1 = TimeSeries.fetch_open_data('H1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)
dL1 = TimeSeries.fetch_open_data('L1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)

n_analyze = Ntime(srate, T)

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('H1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].location, ra, dec, tgps))
tH1 = tgps + dt_ifo

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('L1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].location, ra, dec, tgps))
tL1 = tgps + dt_ifo

In [5]:
tH1, tL1

(1242442967.4318974, 1242442967.4302728)

In [6]:
dH1_cond = bandpass_ds(dH1, t0=tH1, ds=int(fs/srate), trim=0.25, f_min=fmin)
dL1_cond = bandpass_ds(dL1, t0=tL1, ds=int(fs/srate), trim=0.25, f_min=fmin)

In [7]:
psdH = sig.welch(
    dH1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",   
)

psdL = sig.welch(
    dL1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",  
)

In [8]:
dt_np = 1.0 / srate
Nt     = Ntime(srate=srate, T=T)
freqs_np = onp.fft.rfftfreq(Nt, d=dt_np)

fmin_eff = freqs_np[0]  if (fmin is None) else float(fmin)
fmax_eff = freqs_np[-1] if (fmax is None) else float(fmax)

i0 = int(onp.searchsorted(freqs_np, fmin_eff, side="left"))
i1 = int(onp.searchsorted(freqs_np, fmax_eff, side="right"))

freqs = jnp.asarray(freqs_np, dtype=jnp.float64)

# Prior limits
limits = [
    [30.0, 500.0],
    [0., 0.99],
    [0., 4.0],
    [0., 6.283185307179586],
    [0., 4.0],
    [0., 6.283185307179586],
    [-1., 1.],
    [0., 3.141592653589793]]

low  = onp.asarray([pair[0] for pair in limits], dtype=onp.float64)
high = onp.asarray([pair[1] for pair in limits], dtype=onp.float64)

In [9]:
psdH = interp1d_jax(jnp.asarray(psdH[0],dtype=jnp.float64), jnp.asarray(psdH[1],dtype=jnp.float64))
psdL = interp1d_jax(jnp.asarray(psdL[0],dtype=jnp.float64), jnp.asarray(psdL[1],dtype=jnp.float64))

omega_22_r, omega_22_i, omega_33_r, omega_33_i = load_tables(qnm1_path, qnm2_path)
tgps = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps)

rhoH, rhoL = ACFs(srate=srate, T=T, psdH=psdH, psdL=psdL, factor=factor)
covH=toeplitz(rhoH)
covL=toeplitz(rhoL)
L_H=jnp.linalg.cholesky(covH)
L_L=jnp.linalg.cholesky(covL)

resp_H_py = lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].response
resp_L_py = lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].response
resp_H = jnp.asarray(resp_H_py, dtype=jnp.float64)
resp_L = jnp.asarray(resp_L_py, dtype=jnp.float64)

set_detectors(resp_H, resp_L)

In [10]:
MTSUN = lal.MTSUN_SI
tM = MTSUN * 325 # Remnant mass peak

In [11]:
num_shifts = 15

t_shifts = []
H1_data_t_shifted = []
L1_data_t_shifted = []

for jj in range(num_shifts):
    dH1_analysis_data = analysis_data(dH1_cond[1], dH1_cond[0], tH1 + jj*1*tM, n_analyze)
    dL1_analysis_data = analysis_data(dL1_cond[1], dL1_cond[0], tL1 + jj*1*tM, n_analyze)

    t_shifts.append(jj*1*tM)
    H1_data_t_shifted.append(dH1_analysis_data[1])
    L1_data_t_shifted.append(dL1_analysis_data[1])

In [12]:
onp.savetxt(tM_shifted_samples_save_dir + 't_shifts.txt', t_shifts)

In [13]:
low  = jnp.asarray(low,  dtype=jnp.float64)
high = jnp.asarray(high, dtype=jnp.float64)

theta_fixed_sky = jnp.array([ra, dec]) 

In [14]:
logZs = []

for i in range(num_shifts):
    h_H_t = H1_data_t_shifted[i]
    h_L_t = L1_data_t_shifted[i]

    def make_loglik_fn(dataH, dataL, gmst, L_H, L_L,
                    omega_22_r, omega_22_i, omega_33_r, omega_33_i,
                    T, srate, t0):
        def _loglik(theta):
            return ln_likelihood_full_jit(
                dataH=dataH, dataL=dataL, params=theta,
                gmst=gmst, L_H=L_H, L_L=L_L,
                omega_22_r=omega_22_r, omega_22_i=omega_22_i,
                omega_33_r=omega_33_r, omega_33_i=omega_33_i,
                T=T, srate=srate, t0=t0
            )
        return _loglik

    loglik_fn = make_loglik_fn(
        dataH=h_H_t, dataL=h_L_t,
        gmst=gmst, L_H=L_H, L_L=L_L,
        omega_22_r=omega_22_r, omega_22_i=omega_22_i,
        omega_33_r=omega_33_r, omega_33_i=omega_33_i,
        T=T, srate=srate, t0=t0
    )

    def model_fixed_sky():
        theta_free = numpyro.sample("theta", dist.Uniform(low, high)) 
        theta = jnp.concatenate([theta_free[:-1], theta_fixed_sky, jnp.array([theta_free[-1]])], axis=0)
        numpyro.factor("loglike", loglik_fn(theta))

    rng_key = jax.random.PRNGKey(int(seed) ^ 0xABCDEF)
    rng_run, rng_post = jax.random.split(rng_key)
    ns_fixed_sky = NestedSampler(
        model_fixed_sky,
        constructor_kwargs=dict(
            num_live_points=10000,    
            max_samples=500_000,      
            verbose=True,
        ),
        termination_kwargs=dict(
            dlogZ=0.01,            
        ),
    )
    ns_fixed_sky.run(rng_run)
    ns_fixed_sky.print_summary()
    posterior_fixed_sky = ns_fixed_sky.get_samples(rng_post, num_samples=100_000)

    onp.save(tM_shifted_samples_save_dir + 'posterior.' + event_id + '.' + str(i) + '.npy', onp.asarray(posterior_fixed_sky['theta']))
    logZs.append(ns_fixed_sky._results.log_Z_mean)


INFO:jaxns:Number of Markov-chains set to: 10000


Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None, evidence_uncert=None, live_evidence_frac=None, dlogZ=Array(0.01, dtype=float64, weak_type=True), max_samples=Array(500400, dtype=int64), max_num_likelihood_evaluations=None, log_L_contour=None, efficiency_threshold=None, rtol=None, atol=None, peak_XL_frac=None)
-------
Num samples: 5004
Num likelihood evals: 267675
Efficiency: 0.03603266270144159
log(L) contour: -5194.469522457423
log(Z) est.: -822.8954337123578 +- 0.8323820538052353
log(Z | remaining) est.: 4382.287433906799 +- 1.1706185611496602
ESS: 0.5002624165534695

-------
Num samples: 10008
Num likelihood evals: 565520
Efficiency: 0.022989033808213388
log(L) contour: -2223.375159355599
log(Z) est.: -823.3864517287896 +- 0.8296624675319875
log(Z | remaining) est.: 1408.7824051346367 +- 0.9119867949446319
ESS: 0.5048422766039378

-

INFO:jaxns:Number of Markov-chains set to: 10000


Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None, evidence_uncert=None, live_evidence_frac=None, dlogZ=Array(0.01, dtype=float64, weak_type=True), max_samples=Array(500400, dtype=int64), max_num_likelihood_evaluations=None, log_L_contour=None, efficiency_threshold=None, rtol=None, atol=None, peak_XL_frac=None)
-------
Num samples: 5004
Num likelihood evals: 267652
Efficiency: 0.036035517020372596
log(L) contour: -5181.7769495359535
log(Z) est.: -838.1861386720334 +- 0.6389634552401986
log(Z | remaining) est.: 4354.130627069506 +- 1.00277547706586
ESS: 0.9915329795575164

-------
Num samples: 10008
Num likelihood evals: 565331
Efficiency: 0.022950864785282826
log(L) contour: -2223.2578891201433
log(Z) est.: -833.6510854888301 +- 0.5580506348513828
log(Z | remaining) est.: 1398.9664283605443 +- 0.7138063007681236
ESS: 1.3685010521282275



INFO:jaxns:Number of Markov-chains set to: 10000


...

--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 17589780
samples: 155124
phantom samples: 0
likelihood evals / sample: 113.4
phantom fraction (%): 0.0%
--------
logZ=-822.192 +- 0.037
max(logL)=-808.615
H=-9.53
ESS=21328
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 312.0 +- 71.0 | 228.0 / 299.0 / 417.0 | 345.0 | 345.0
theta[1]: 0.42 +- 0.25 | 0.08 / 0.41 / 0.77 | 0.89 | 0.89
theta[2]: 0.35 +- 0.31 | 0.12 / 0.25 / 0.7 | 0.39 | 0.39
theta[3]: 3.1 +- 1.8 | 0.5 / 3.2 / 5.7 | 4.6 | 4.6
theta[4]: 0.12 +- 0.12 | 0.01 / 0.07 / 0.3 | 0.06 | 0.06
theta[5]: 3.1 +- 1.8 | 0.6 / 3.1 / 5.5 | 1.1 | 1.1
theta[6]: -0.09 +- 0.34 | -0.51 / -0.08 / 0.32 | -0.15 | -0.15
theta[7]: 1.51 +- 0.87 | 0.39 / 1.62 / 2.68 | 0.56 | 0.56
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCond

INFO:jaxns:Number of Markov-chains set to: 10000


--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 18326482
samples: 160128
phantom samples: 0
likelihood evals / sample: 114.4
phantom fraction (%): 0.0%
--------
logZ=-821.78 +- 0.037
max(logL)=-808.051
H=-9.17
ESS=21026
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 329.0 +- 81.0 | 233.0 / 314.0 / 455.0 | 339.0 | 339.0
theta[1]: 0.37 +- 0.25 | 0.06 / 0.34 / 0.73 | 0.88 | 0.88
theta[2]: 0.38 +- 0.36 | 0.12 / 0.26 / 0.77 | 0.36 | 0.36
theta[3]: 3.1 +- 1.8 | 0.6 / 3.2 / 5.5 | 5.1 | 5.1
theta[4]: 0.13 +- 0.12 | 0.01 / 0.09 / 0.33 | 0.07 | 0.07
theta[5]: 3.1 +- 1.8 | 0.6 / 3.1 / 5.6 | 1.7 | 1.7
theta[6]: -0.05 +- 0.34 | -0.48 / -0.04 / 0.37 | -0.16 | -0.16
theta[7]: 1.48 +- 0.89 | 0.38 / 1.54 / 2.67 | 0.58 | 0.58
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCon

INFO:jaxns:Number of Markov-chains set to: 10000


--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 18060484
samples: 155124
phantom samples: 0
likelihood evals / sample: 116.4
phantom fraction (%): 0.0%
--------
logZ=-824.978 +- 0.037
max(logL)=-811.69
H=-9.3
ESS=20676
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 367.0 +- 93.0 | 240.0 / 390.0 / 478.0 | 345.0 | 345.0
theta[1]: 0.31 +- 0.23 | 0.05 / 0.26 / 0.66 | 0.93 | 0.93
theta[2]: 0.48 +- 0.44 | 0.13 / 0.34 / 1.02 | 0.26 | 0.26
theta[3]: 3.2 +- 1.8 | 0.7 / 3.2 / 5.6 | 2.4 | 2.4
theta[4]: 0.2 +- 0.17 | 0.02 / 0.16 / 0.44 | 0.08 | 0.08
theta[5]: 3.0 +- 1.7 | 0.7 / 3.1 / 5.3 | 5.2 | 5.2
theta[6]: -0.04 +- 0.37 | -0.51 / -0.03 / 0.43 | -0.17 | -0.17
theta[7]: 1.46 +- 0.87 | 0.38 / 1.49 / 2.61 | 2.19 | 2.19
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondi

In [22]:
logZs

[Array(-821.81065965, dtype=float64),
 Array(-820.13953815, dtype=float64),
 Array(-819.42201269, dtype=float64),
 Array(-821.73244571, dtype=float64),
 Array(-825.8979573, dtype=float64),
 Array(-826.72181622, dtype=float64),
 Array(-822.26985019, dtype=float64),
 Array(-824.73515268, dtype=float64),
 Array(-825.77356401, dtype=float64),
 Array(-823.81881726, dtype=float64),
 Array(-828.29653316, dtype=float64),
 Array(-821.98770965, dtype=float64),
 Array(-821.40924183, dtype=float64),
 Array(-824.67060764, dtype=float64),
 Array(-825.77353742, dtype=float64)]